## Розширений фазовий простір

Ноутбук містить інтерактивні 3D-побудови для задач 3.7 і 3.6.

In [1]:
# Якщо чогось бракує — розкоментуйте цей рядок і виконайте
# %pip install numpy scipy plotly ipywidgets

import numpy as np
import plotly.graph_objects as go
from scipy.integrate import solve_ivp
from ipywidgets import interact, IntSlider, Checkbox, FloatSlider


def P(x, y):
    return (2.0 * x - y) * (x - 2.0)


def Q(x, y):
    return x * y - 2.0


def rhs(t, z):
    x, y = z
    return [P(x, y), Q(x, y)]


M1 = np.array([2.0, 1.0])
M2 = np.array([1.0, 2.0])
M3 = np.array([-1.0, -2.0])


def integrate(x0, y0, t_span=(0.0, 6.0), max_step=0.05, clip=8.0):
    def event_out(t, z):
        return clip - max(abs(z[0]), abs(z[1]))
    event_out.terminal = True

    sol = solve_ivp(
        rhs, t_span, [x0, y0],
        max_step=max_step, rtol=1e-8, atol=1e-10,
        events=event_out,
    )
    return sol.t, sol.y[0], sol.y[1]


def _add_arrows(fig, x_all, y_all, t_all, color, density, sizeref=0.25):
    # Додає go.Cone-стрілки вздовж траєкторії.
    # density — приблизна кількість стрілок на криву.
    # Дотичний вектор нормалізуємо до одиничної довжини, щоб конуси
    # мали однаковий розмір незалежно від швидкості руху.
    n_pts = len(t_all)
    if n_pts < 4 or density <= 0:
        return
    step = max(1, n_pts // density)
    idx = np.arange(0, n_pts - 1, step)
    if len(idx) == 0:
        return
    dx = x_all[idx + 1] - x_all[idx]
    dy = y_all[idx + 1] - y_all[idx]
    dt = t_all[idx + 1] - t_all[idx]
    norm = np.sqrt(dx * dx + dy * dy + dt * dt)
    mask = norm > 1e-12
    if not np.any(mask):
        return
    ux = dx[mask] / norm[mask]
    uy = dy[mask] / norm[mask]
    uz = dt[mask] / norm[mask]
    fig.add_trace(go.Cone(
        x=x_all[idx][mask], y=y_all[idx][mask], z=t_all[idx][mask],
        u=ux, v=uy, w=uz,
        sizemode="absolute", sizeref=sizeref,
        anchor="tail",
        showscale=False,
        colorscale=[[0, color], [1, color]],
        hoverinfo="skip",
    ))


def build_extended_figure(starts, t_max=4.0, t_min=-2.0, show_shadow=True,
                          arrows=False, arrow_density=6, clip=8.0):
    fig = go.Figure()

    for i, (x0, y0) in enumerate(starts):
        t_f, xf, yf = integrate(x0, y0, (0.0, t_max), clip=clip)
        t_b, xb, yb = integrate(x0, y0, (0.0, t_min), clip=clip)
        t_all = np.concatenate([t_b[::-1], t_f[1:]])
        x_all = np.concatenate([xb[::-1], xf[1:]])
        y_all = np.concatenate([yb[::-1], yf[1:]])

        color = f"hsl({(i * 43) % 360}, 70%, 45%)"
        fig.add_trace(go.Scatter3d(
            x=x_all, y=y_all, z=t_all,
            mode="lines",
            line=dict(width=3, color=color),
            showlegend=False,
            hovertemplate="x=%{x:.2f}<br>y=%{y:.2f}<br>t=%{z:.2f}<extra></extra>",
        ))

        if arrows:
            _add_arrows(fig, x_all, y_all, t_all, color, arrow_density, sizeref=0.3)

        if show_shadow:
            fig.add_trace(go.Scatter3d(
                x=x_all, y=y_all, z=np.full_like(t_all, t_min),
                mode="lines",
                line=dict(width=1, color="rgba(70,70,70,0.35)"),
                showlegend=False,
                hoverinfo="skip",
            ))

    for point, name, color in [
        (M1, "M1", "crimson"),
        (M2, "M2", "darkgreen"),
        (M3, "M3", "slateblue"),
    ]:
        fig.add_trace(go.Scatter3d(
            x=[point[0], point[0]], y=[point[1], point[1]], z=[t_min, t_max],
            mode="lines", line=dict(color=color, width=3, dash="dash"),
            name=name,
        ))

    fig.update_layout(
        title="## 3. Панорама: сітка стартових точок",
        scene=dict(
            xaxis_title="x", yaxis_title="y", zaxis_title="t",
            xaxis=dict(range=[-4, 5]), yaxis=dict(range=[-4, 5]), zaxis=dict(range=[t_min, t_max]),
            aspectmode="cube",
        ),
        width=920, height=720, margin=dict(l=0, r=0, t=40, b=0),
    )
    return fig


@interact(
    n=IntSlider(value=6, min=3, max=10, step=1, description="n×n"),
    t_max=FloatSlider(value=4.0, min=1.0, max=8.0, step=0.5, description="t_max"),
    t_min=FloatSlider(value=-2.0, min=-5.0, max=0.0, step=0.5, description="t_min"),
    shadow=Checkbox(value=True, description="Тінь"),
    arrows=Checkbox(value=True, description="Стрілки"),
    arrow_density=IntSlider(value=6, min=1, max=20, step=1, description="к-ть/крива"),
)
def grid_view(n, t_max, t_min, shadow, arrows, arrow_density):
    xs = np.linspace(-3.0, 4.0, n)
    ys = np.linspace(-3.0, 4.0, n)
    starts = []
    for x in xs:
        for y in ys:
            if any(np.hypot(x - m[0], y - m[1]) < 0.2 for m in (M1, M2, M3)):
                continue
            starts.append((float(x), float(y)))

    fig = build_extended_figure(
        starts, t_max=t_max, t_min=t_min, show_shadow=shadow,
        arrows=arrows, arrow_density=arrow_density,
    )
    fig.show()

interactive(children=(IntSlider(value=6, description='n×n', max=10, min=3), FloatSlider(value=4.0, description…

In [2]:
# 3.6: розширений фазовий простір для системи
# x' = ln(2 - y^2), y' = e^x - e^y

SQRT2 = np.sqrt(2.0)
M1_36 = np.array([1.0, 1.0])
M2_36 = np.array([-1.0, -1.0])


def P36(x, y):
    arg = 2.0 - y * y
    return np.log(arg)


def Q36(x, y):
    return np.exp(x) - np.exp(y)


def rhs36(t, z):
    x, y = z
    return [P36(x, y), Q36(x, y)]


def integrate36(x0, y0, t_span=(0.0, 4.0), max_step=0.04):
    # зупинка на межі області означення |y| < sqrt(2)
    def event_up(t, z):
        return (SQRT2 - 1e-4) - z[1]

    def event_down(t, z):
        return z[1] + (SQRT2 - 1e-4)

    event_up.terminal = True
    event_down.terminal = True

    sol = solve_ivp(
        rhs36,
        t_span,
        [x0, y0],
        max_step=max_step,
        rtol=1e-8,
        atol=1e-10,
        events=[event_up, event_down],
    )
    return sol.t, sol.y[0], sol.y[1]


def build_extended_figure_36(starts, t_min=-2.0, t_max=4.0, show_shadow=True,
                             arrows=False, arrow_density=6):
    fig = go.Figure()

    for i, (x0, y0) in enumerate(starts):
        # пропускаємо некоректні старти поза |y|<sqrt(2)
        if abs(y0) >= SQRT2 - 1e-4:
            continue

        t_f, xf, yf = integrate36(x0, y0, (0.0, t_max))
        t_b, xb, yb = integrate36(x0, y0, (0.0, t_min))

        t_all = np.concatenate([t_b[::-1], t_f[1:]])
        x_all = np.concatenate([xb[::-1], xf[1:]])
        y_all = np.concatenate([yb[::-1], yf[1:]])

        color = f"hsl({(i * 47) % 360}, 70%, 45%)"
        fig.add_trace(go.Scatter3d(
            x=x_all,
            y=y_all,
            z=t_all,
            mode="lines",
            line=dict(width=3, color=color),
            showlegend=False,
            hovertemplate="x=%{x:.2f}<br>y=%{y:.2f}<br>t=%{z:.2f}<extra></extra>",
        ))

        if arrows:
            _add_arrows(fig, x_all, y_all, t_all, color, arrow_density, sizeref=0.18)

        if show_shadow:
            fig.add_trace(go.Scatter3d(
                x=x_all,
                y=y_all,
                z=np.full_like(t_all, t_min),
                mode="lines",
                line=dict(width=1, color="rgba(70,70,70,0.35)"),
                showlegend=False,
                hoverinfo="skip",
            ))

    # вертикалі через особливі точки
    for point, name, color in [
        (M1_36, "M1 (focus)", "crimson"),
        (M2_36, "M2 (saddle)", "darkgreen"),
    ]:
        fig.add_trace(go.Scatter3d(
            x=[point[0], point[0]],
            y=[point[1], point[1]],
            z=[t_min, t_max],
            mode="lines",
            line=dict(color=color, width=3, dash="dash"),
            name=name,
        ))

    # межі області означення y = +-sqrt(2)
    xs = np.linspace(-2.5, 2.5, 120)
    for yb in [SQRT2, -SQRT2]:
        fig.add_trace(go.Scatter3d(
            x=xs,
            y=np.full_like(xs, yb),
            z=np.full_like(xs, t_min),
            mode="lines",
            line=dict(color="black", width=2, dash="dot"),
            showlegend=False,
            hoverinfo="skip",
        ))

    fig.update_layout(
        title="3.6: розширений фазовий простір (x, y, t)",
        scene=dict(
            xaxis_title="x",
            yaxis_title="y",
            zaxis_title="t",
            xaxis=dict(range=[-2.5, 2.5]),
            yaxis=dict(range=[-1.45, 1.45]),
            zaxis=dict(range=[t_min, t_max]),
            aspectmode="cube",
        ),
        width=920,
        height=720,
        margin=dict(l=0, r=0, t=40, b=0),
    )
    return fig


@interact(
    n=IntSlider(value=6, min=3, max=10, step=1, description="n×n"),
    t_max=FloatSlider(value=4.0, min=1.0, max=8.0, step=0.5, description="t_max"),
    t_min=FloatSlider(value=-2.0, min=-5.0, max=0.0, step=0.5, description="t_min"),
    shadow=Checkbox(value=True, description="Тінь"),
    arrows=Checkbox(value=True, description="Стрілки"),
    arrow_density=IntSlider(value=6, min=1, max=20, step=1, description="к-ть/крива"),
)
def grid_view_36(n, t_max, t_min, shadow, arrows, arrow_density):
    xs = np.linspace(-2.4, 2.4, n)
    ys = np.linspace(-1.35, 1.35, n)
    starts = []
    for x in xs:
        for y in ys:
            if any(np.hypot(x - m[0], y - m[1]) < 0.12 for m in (M1_36, M2_36)):
                continue
            starts.append((float(x), float(y)))

    fig = build_extended_figure_36(
        starts, t_min=t_min, t_max=t_max, show_shadow=shadow,
        arrows=arrows, arrow_density=arrow_density,
    )
    fig.show()

interactive(children=(IntSlider(value=6, description='n×n', max=10, min=3), FloatSlider(value=4.0, description…

5й слайд лекції 5


у 2му рівнянні мінус